<a href="https://colab.research.google.com/github/CPTR295/NLP-Basic-Using-Pytorch/blob/main/Classification_Surnames_Using_CNN_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Pre-Processing

In [74]:
#Import Dataset
import kagglehub
path = kagglehub.dataset_download('alenic/surname-dataset-classification')
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'surname-dataset-classification' dataset.
Path to dataset files: /kaggle/input/surname-dataset-classification


In [75]:
!cp '/kaggle/input/surname-dataset-classification' '/content/'

cp: -r not specified; omitting directory '/kaggle/input/surname-dataset-classification'


In [76]:
import pandas as pd
import numpy as np
import collections
import re
from argparse import Namespace

In [77]:
args = Namespace(
    raw_dataset_csv = '/content/surname-dataset-classification/surname-nationality.csv',
    train_proportion = 0.7,
    val_proportion = 0.15,
    test_proportion = 0.15,
    output_munged_csv = '/content/surnames_with_splits.csv',
    seed = 1337
)

In [78]:
data = pd.read_csv(args.raw_dataset_csv,header=0)

In [79]:
data.head()

,rank,surname,nationality,frequency,census_count
0,1.0,Tesfaye,Ethiopian,0.011905,1167260
1,2.0,Mohammed,Ethiopian,0.011111,1084839
2,3.0,Getachew,Ethiopian,0.009174,895366
3,4.0,Abebe,Ethiopian,0.008475,825501
4,5.0,Girma,Ethiopian,0.008403,822765


In [80]:
surnames = data[['surname','nationality']]

In [81]:
surnames.head()

,surname,nationality
0,Tesfaye,Ethiopian
1,Mohammed,Ethiopian
2,Getachew,Ethiopian
3,Abebe,Ethiopian
4,Girma,Ethiopian


In [82]:
set(surnames.nationality)

{'Algerian',
 'Arabic',
 'Brazilian',
 'Chilean',
 'Chinese',
 'Czech',
 'Dutch',
 'English',
 'Ethiopian',
 'Finnish',
 'French',
 'German',
 'Greek',
 'Honduran',
 'Indian',
 'Irish',
 'Italian',
 'Japanese',
 'Korean',
 'Malaysian',
 'Mexican',
 'Moroccan',
 'Nepalese',
 'Nicaraguan',
 'Nigerian',
 'Palestinian',
 'Papua New Guinean',
 'Peruvian',
 'Polish',
 'Portuguese',
 'Russian',
 'Scottish',
 'South African',
 'Spanish',
 'Ukrainian',
 'Venezuelan',
 'Vietnamese'}

In [83]:
#splitting data by nationality
by_nationality  = collections.defaultdict(list)
for _,row in surnames.iterrows():
  by_nationality[row.nationality].append(row.to_dict())

In [84]:
len(by_nationality['Dutch']) #Example  lenght

1269

In [85]:
by_nationality.keys()

dict_keys(['Ethiopian', 'Honduran', 'Nigerian', 'Malaysian', 'Chilean', 'Portuguese', 'Papua New Guinean', 'Algerian', 'Brazilian', 'Venezuelan', 'Ukrainian', 'South African', 'Nicaraguan', 'Moroccan', 'Finnish', 'Mexican', 'Palestinian', 'Nepalese', 'Peruvian', 'Dutch', 'Arabic', 'Irish', 'Spanish', 'French', 'German', 'English', 'Korean', 'Indian', 'Vietnamese', 'Scottish', 'Japanese', 'Polish', 'Greek', 'Czech', 'Italian', 'Russian', 'Chinese'])

In [86]:
#Split the data
final_list=[]
np.random.seed(args.seed)

for _,item_list in sorted(by_nationality.items()):
  np.random.shuffle(item_list)
  n=len(item_list)
  n_train = int(args.train_proportion*n)
  n_val = int(args.val_proportion*n)
  n_test = int(args.test_proportion*n)
  # print(type(item_list))
  # print(type(item_list[0]))
  for item in item_list[:n_train]:
    item['split']='train'
  for item in item_list[n_train:n_train+n_val]:
    item['split']='val'
  for item in item_list[n_train+n_val:]:
    item['split']='test'

  final_list.extend(item_list)

In [87]:
final_surnames = pd.DataFrame(final_list)

In [88]:
final_surnames.head()

,surname,nationality,split
0,Aziez,Algerian,train
1,Saadi,Algerian,train
2,Bensalem,Algerian,train
3,Abidat,Algerian,train
4,Belabed,Algerian,train


In [89]:
final_surnames.split.value_counts()

,count
split,
train,25358
test,5460
val,5423


In [90]:
final_surnames.to_csv(args.output_munged_csv,index=False)

Classification Surnames With CNN

In [91]:
from argparse import Namespace
from collections import Counter
import json
import os
import string

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm_notebook

Data Vectorization Classes

In [92]:
#Vocabulary Class
class Vocabulary(object):
  #Class to process test and extract vocabulary for mapping
  def __init__(self,token_to_idx=None,add_unk=True,unk_token="<UNK"):
    #token_to_idx - dict - a map for token to indices
    if token_to_idx is None:
      token_to_idx={}
    self._token_to_idx = token_to_idx

    self._idx_to_token = {idx:token for token,idx in self._token_to_idx.items()}

    self._add_unk = add_unk
    self._unk_token = unk_token

    self.unk_index=-1
    if add_unk:
      self.unk_index = self.add_token(unk_token)

  def to_serializable(self):
    return {'token_to_idx':self._token_to_idx,'add_unk':self._add_unk,'unk_token':self._unk_token}

  @classmethod
  def from_serializable(cls,contents):
    return cls(**contents)

  def add_token(self,token):
    try:
      index=self._token_to_idx[token]
    except KeyError:
      index=len(self._token_to_idx)
      self._token_to_idx[token]=index #Because index starts with zero
      self._idx_to_token[index]=token
    return index

  def add_many(self,tokens):
    return [self.add_token(token) for token in tokens]

  def lookup_token(self,token):
    if self.unk_index >= 0:
        return self._token_to_idx.get(token, self.unk_index)
    else:
        return self._token_to_idx[token]

  def lookup_index(self,index):
    if index not in self._idx_to_token:
      raise KeyError("the index (%d) is not in the Vocabulary" % index)
    return self._idx_to_token[index]

  def __str__(self) -> str:
    return "<Vocabulary(size=%d)>" % len(self)

  def __len__(self):
    return len(self._token_to_idx)

In [93]:
#Vecotrizer Class
class SurnameVectorizer(object):
  #Coordinates Vocabularies and put them to use
  def __init__(self,surname_vocab,nationality_vocab,max_surname_length):
    self.surname_vocab = surname_vocab
    self.nationality_vocab = nationality_vocab
    self.max_surname_length = max_surname_length

  def vecotrize(self,surname):
    vocab = self.surname_vocab
    one_hot_matrix_size = (len(vocab),self.max_surname_length)
    one_hot_matrix = np.zeros(one_hot_matrix_size,dtype=np.float32)
    for position_index,character in enumerate(surname):
      character_index = vocab.lookup_token(character)
      one_hot_matrix[character_index][position_index]=1
    return one_hot_matrix

  @classmethod
  def from_dataframe(cls,surname_df):
    surname_vocab = Vocabulary(unk_token="@")
    nationality_vocab = Vocabulary(add_unk=False)
    max_surname_length = 0
    for index,row in surname_df.iterrows():
      max_surname_length = max(max_surname_length,len(row.surname))
      for letter in row.surname:
        surname_vocab.add_token(letter)
      nationality_vocab.add_token(row.nationality)
    return cls(surname_vocab,nationality_vocab,max_surname_length)

  @classmethod
  def from_serializable(cls,contents):
    surname_vocab = Vocabulary.from_serializable(contents['surname_vocab'])
    nationality_vocab = Vocabulary.from_serializable(contents['nationality_vocab'])
    return cls(surname_vocab=surname_vocab,nationality_vocab=nationality_vocab,max_surname_length=contents['max_surname_length'])

  def to_serializable(self):
    return {'surname_vocab':self.surname_vocab.to_serializable(),'nationality_vocab':self.nationality_vocab.to_serializable(),'max_surname_length':self.max_surname_length}

In [94]:
#Dataset
class SurnameDataset(Dataset):
  def __init__(self,surname_df,vectorizer):
    self.surname_df = surname_df
    self._vectorizer = vectorizer

    self.train_df = self.surname_df[self.surname_df['split']=='train']
    self.train_size = len(self.train_df)

    self.val_df = self.surname_df[self.surname_df['split']=='val']
    self.validation_size = len(self.val_df)

    self.test_df = self.surname_df[self.surname_df['split']=='test']
    self.test_size = len(self.test_df)

    self.lookup_dict = {'train':(self.train_df,self.train_size),
                         'val':(self.val_df,self.validation_size),
                         'test':(self.test_df,self.test_size)}
    self.set_split('train')

    class_counts = surname_df.nationality.value_counts().to_dict()
    def sort_key(item):
      return self._vectorizer.nationality_vocab.lookup_token(item[0])
    sorted_counts = sorted(class_counts.items(),key=sort_key)
    frequencies = [count for _,count in sorted_counts]
    self.class_weights = 1.0 / torch.tensor(frequencies,dtype=torch.float32)

  @classmethod
  def load_dataset_and_make_vectorizer(cls,surname_csv):
    surname_df = pd.read_csv(surname_csv)
    train_surname_df = surname_df[surname_df['split']=='train']
    return cls(surname_df,SurnameVectorizer.from_dataframe(train_surname_df))

  @classmethod
  def load_dataset_and_load_vectorizer(cls,surname_csv,vectorizer_filepath):
    surname_df = pd.read_csv(surname_csv)
    vectorizer = cls.load_vectorizer_only(vectorizer_filepath)
    return cls(surname_df,vectorizer)

  @classmethod
  def load_vectorizer_only(cls,vectorizer_filepath):
    with open(vectorizer_filepath) as fp:
      return SurnameVectorizer.from_serializable(json.load(fp))

  def save_vectorizer(self,vectorizer_filepath):
    with open(vectorizer_filepath,'w') as fp:
      json.dump(self._vectorizer.to_serializable(),fp)

  def get_vectorizer(self):
    return self._vectorizer

  def set_split(self,split='train'):
    self._target_split = split
    self._target_df,self._target_size = self.lookup_dict[split]

  def __getitem__(self, index):
    row = self._target_df.iloc[index]
    surname_vector = self._vectorizer.vecotrize(row.surname)
    nationality_index = self._vectorizer.nationality_vocab.lookup_token(row.nationality)
    return {'x_surname':surname_vector,'y_nationality':nationality_index}

  def __len__(self):
        return self._target_size

  def get_num_batches(self,batch_size):
    return len(self) // batch_size

In [95]:
def generator_batches(dataset,batch_size,shuffle=True,drop_last=True,device='cpu'):
  dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last)
  for data_dict in dataloader:
    out_data_dict = {}
    for name,tensor in data_dict.items():
      out_data_dict[name] = data_dict[name].to(device)
    yield out_data_dict

In [168]:
#The MODEL
class SurnameClassifier(nn.Module):
  def __init__(self,initial_num_channels,num_classes,num_channels):
    super(SurnameClassifier,self).__init__()
    self.convnet = nn.Sequential(
        nn.Conv1d(in_channels=initial_num_channels,out_channels=num_channels,kernel_size=3),
        nn.ELU(),
        nn.Conv1d(in_channels=num_channels,out_channels=num_channels,kernel_size=3,stride=2),
        nn.ELU(),
        nn.Conv1d(in_channels=num_channels,out_channels=num_channels,kernel_size=3,stride=2),
        nn.ELU(),
        nn.Conv1d(in_channels=num_channels,out_channels=num_channels,kernel_size=3),
        nn.ELU()
    )
    self.fc = nn.Linear(num_channels,num_classes)  #256x37

  def forward(self,x_surname,apply_softmax=False):
    features = self.convnet(x_surname).squeeze(dim=2) #32768x2 Sqeeze is not working as intended
    features = features.mean(dim=2)
    #print(features.shape) # [128, 256, 2]
    prediction_vector = self.fc(features)
    if apply_softmax:
      prediction_vector = F.softmax(prediction_vector,dim=1)
    return prediction_vector

In [169]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters     : {total:,}")
    print(f"Trainable parameters : {trainable:,}")

In [170]:
model = SurnameClassifier(
    initial_num_channels=26,
    num_classes=18,
    num_channels=256
)

count_parameters(model)

Total parameters     : 615,442
Trainable parameters : 615,442


In [171]:
def make_train_state(args):
    return {'stop_early': False,
            'early_stopping_step': 0,
            'early_stopping_best_val': 1e8,
            'learning_rate': args.learning_rate,
            'epoch_index': 0,
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'test_loss': -1,
            'test_acc': -1,
            'model_filename': args.model_state_file}

In [172]:
def update_train_state(args,model,train_state):
  if train_state['epoch_index'] == 0:
    torch.save(model.state_dict(),train_state['model_filename'])
    train_state['stop_early'] = False
  elif train_state['epoch_index'] >= 1:
    loss_tm1,loss_t = train_state['val_loss'][-2:]
    if loss_t >= train_state['early_stopping_best_val']:
      train_state['early_stopping_step'] += 1
    else:
      if loss_t < train_state['early_stopping_best_val']:
        torch.save(model.state_dict(),train_state['model_filename'])
      train_state['early_stopping_step'] = 0
    train_state['stop_early'] = train_state['early_stopping_step'] >= args.early_stopping_criteria
  return train_state

In [173]:
def compute_accuracy(y_pred, y_target):
    y_pred_indices = y_pred.max(dim=1)[1]
    n_correct = torch.eq(y_pred_indices, y_target).sum().item()
    return n_correct / len(y_pred_indices) * 100

In [174]:
args = Namespace(
    surname_csv = '/content/surnames_with_splits.csv',
    vectorizer_file = 'vectorizer.json',
    model_state_file = 'model.pth',
    save_dir = '/content/',
    hidden_dim=100,
    num_channels=256,
    seed=1337,
    learning_rate=0.001,
    batch_size=128,
    num_epochs=100,
    early_stopping_criteria=5,
    dropout_p=0.1,
    cuda=True,
    catch_keyboard_interrupt=True,
    reload_from_files=False,
    expand_filepaths_to_save_dir=True

)

In [175]:
def set_seed_everywhere(seed,cuda):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if cuda:
    torch.cuda.manual_seed_all(seed)

In [176]:
def handle_dirs(dirpath):
  if not os.path.exists(dirpath):
    os.makedirs(dirpath)

In [177]:
if args.expand_filepaths_to_save_dir:
  args.vectorizer_file = os.path.join(args.save_dir,args.vectorizer_file)
  args.model_state_file = os.path.join(args.save_dir,args.model_state_file)
  print("Expanded filepaths: ")
  print("\t{}".format(args.vectorizer_file))
  print("\t{}".format(args.model_state_file))

Expanded filepaths: 
	/content/vectorizer.json
	/content/model.pth


In [178]:
if not torch.cuda.is_available():
  args.cuda=False
args.device = torch.device("cuda" if args.cuda else "cpu")
print(args.device)

cuda


In [179]:
set_seed_everywhere(args.seed, args.cuda)
handle_dirs(args.save_dir)

In [180]:
if args.reload_from_files:
  print('Reloading')
  dataset = SurnameDataset.load_dataset_and_load_vectorizer(args.surname_csv,args.vectorizer_file)
else:
  print('Creating')
  dataset = SurnameDataset.load_dataset_and_make_vectorizer(args.surname_csv)
  dataset.save_vectorizer(args.vectorizer_file)

Creating


In [181]:
vectorizer = dataset.get_vectorizer()
classifier = SurnameClassifier(initial_num_channels=len(vectorizer.surname_vocab),num_classes=len(vectorizer.nationality_vocab),num_channels=args.num_channels)


In [182]:
classifier = classifier.to(device=args.device)
dataset.class_weights = dataset.class_weights.to(args.device)

In [183]:
loss_func = nn.CrossEntropyLoss(dataset.class_weights)
optimizer = optim.Adam(classifier.parameters(),lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,mode='min',factor=0.5,patience=1)
train_state = make_train_state(args)

In [184]:
count_parameters(classifier)

Total parameters     : 644,133
Trainable parameters : 644,133


In [185]:
try:
  for epoch_index in range(args.num_epochs):
    train_state['epoch_index']=epoch_index
    dataset.set_split('train')
    batch_generator = generator_batches(dataset=dataset,batch_size=args.batch_size,device=args.device)
    classifier.train()
    running_loss = 0.0
    running_acc=0.0
    for batch_index,batch_dict in enumerate(batch_generator,start=1):
      optimizer.zero_grad()
      y_pred = classifier(x_surname=batch_dict['x_surname'])
      loss = loss_func(y_pred,batch_dict['y_nationality'])
      loss.backward()
      optimizer.step()
      running_loss += (loss.item()-running_loss)/(batch_index)
      acc_t = compute_accuracy(y_pred,batch_dict['y_nationality'])
      running_acc += (acc_t-running_acc)/(batch_index)
    train_state['train_loss'].append(running_loss)
    train_state['train_acc'].append(running_acc)

    #Validation
    dataset.set_split('val')
    batch_generator = generator_batches(dataset=dataset,batch_size=args.batch_size,device=args.device)
    classifier.eval()
    running_loss = 0.0
    running_acc=0.0
    with torch.no_grad():
      for batch_index,batch_dict in enumerate(batch_generator,start=1):
        y_pred = classifier(x_surname=batch_dict['x_surname'])
        loss = loss_func(y_pred,batch_dict['y_nationality'])
        running_loss += (loss.item()-running_loss)/(batch_index)
        acc_t = compute_accuracy(y_pred,batch_dict['y_nationality'])
        running_acc += (acc_t-running_acc)/(batch_index)
      train_state['val_loss'].append(running_loss)
      train_state['val_acc'].append(running_acc)

    train_state = update_train_state(args=args,model=classifier,train_state=train_state)
    scheduler.step(running_loss)

    print(
            f"Epoch [{epoch_index+1}/{args.num_epochs}] | "
            f"Train Loss: {train_state['train_loss'][-1]:.4f} | "
            f"Train Acc: {train_state['train_acc'][-1]:.2f}% | "
            f"Val Loss: {train_state['val_loss'][-1]:.4f} | "
            f"Val Acc: {train_state['val_acc'][-1]:.2f}%"
        )

    if train_state['stop_early']:
      print("Early stopping triggered.")
      break



except KeyboardInterrupt:
  print('Exiting Loop')

Epoch [1/100] | Train Loss: 3.0894 | Train Acc: 17.96% | Val Loss: 2.6653 | Val Acc: 25.35%
Epoch [2/100] | Train Loss: 2.5419 | Train Acc: 30.17% | Val Loss: 2.4999 | Val Acc: 30.80%
Epoch [3/100] | Train Loss: 2.3348 | Train Acc: 34.65% | Val Loss: 2.3731 | Val Acc: 36.61%
Epoch [4/100] | Train Loss: 2.1818 | Train Acc: 37.56% | Val Loss: 2.2827 | Val Acc: 36.72%
Epoch [5/100] | Train Loss: 2.0405 | Train Acc: 40.25% | Val Loss: 2.3039 | Val Acc: 37.41%
Epoch [6/100] | Train Loss: 1.9297 | Train Acc: 42.03% | Val Loss: 2.2297 | Val Acc: 40.53%
Epoch [7/100] | Train Loss: 1.8157 | Train Acc: 44.40% | Val Loss: 2.2315 | Val Acc: 42.50%
Epoch [8/100] | Train Loss: 1.7206 | Train Acc: 46.00% | Val Loss: 2.2312 | Val Acc: 41.39%
Epoch [9/100] | Train Loss: 1.5076 | Train Acc: 50.77% | Val Loss: 2.1720 | Val Acc: 44.66%
Epoch [10/100] | Train Loss: 1.4215 | Train Acc: 52.72% | Val Loss: 2.2213 | Val Acc: 42.65%
Epoch [11/100] | Train Loss: 1.3631 | Train Acc: 53.79% | Val Loss: 2.2395 | Va

In [188]:
classifier.load_state_dict(torch.load(train_state['model_filename']))
classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)
loss_func = nn.CrossEntropyLoss(dataset.class_weights)

dataset.set_split('test')
batch_generator = generator_batches(dataset,batch_size=args.batch_size,device=args.device)
running_loss = 0.0
running_acc = 0.0
classifier.eval()
for batch_index, batch_dict in enumerate(batch_generator):
    # compute the output
    y_pred =  classifier(batch_dict['x_surname'])

    # compute the loss
    loss = loss_func(y_pred, batch_dict['y_nationality'])
    loss_t = loss.item()
    running_loss += (loss_t - running_loss) / (batch_index + 1)

    # compute the accuracy
    acc_t = compute_accuracy(y_pred, batch_dict['y_nationality'])
    running_acc += (acc_t - running_acc) / (batch_index + 1)

train_state['test_loss'] = running_loss
train_state['test_acc'] = running_acc

In [189]:
print("Test loss: {};".format(train_state['test_loss']))
print("Test Accuracy: {}".format(train_state['test_acc']))

Test loss: 2.489532232284546;
Test Accuracy: 45.85193452380951


In [192]:
def predict_nationality(surname, classifier, vectorizer):
  vectorized_surname = vectorizer.vectorize(surname)
  vectorized_surname = torch.tensor(vectorized_surname).unsqueeze(0)
  result = classifier(vectorized_surname, apply_softmax=True)

  probability_values, indices = result.max(dim=1)
  index = indices.item()

  predicted_nationality = vectorizer.nationality_vocab.lookup_index(index)
  probability_value = probability_values.item()

  return {'nationality': predicted_nationality, 'probability': probability_value}